# 02 · Limpieza y corrección de datos

**Etapa 2 del pipeline: limpieza** · Responsable: **Integrante 2** (limpieza y corrección de datos)

| | |
|---|---|
| **Lee** | `data/raw/talento_peru_ofertas.csv` (validado con el contrato `raw`) |
| **Escribe** | `data/interim/talento_peru_limpio.csv` y `docs/decisiones/02_limpieza.csv` |
| **Valida** | Contrato `limpio`: id único, sin nulos en los 8 campos principales, vacantes ≥ 1, remuneración > 0 (y ≥ `config.UMBRAL_REMUNERACION_MINIMA`), fecha de fin ≥ fecha de inicio, ningún puesto que sea solo un número |
| **Criterio de la rúbrica** | Preprocesamiento de datos (20%): limpieza |

**Objetivo:** identificar los problemas del dataset original y corregirlos con una justificación para cada decisión.

> **Plantilla preparada por el Integrante 1.** Mantener el encabezado, la celda de preparación y la sección final de guardado: así el notebook encaja en `run_pipeline.py`. Antes de cada commit: *Kernel → Restart & Run All*.

## Preparación

In [ ]:
import sys
from pathlib import Path

# Ubica la raíz del repositorio (la carpeta que contiene src/config.py),
# así el notebook funciona desde notebooks/, desde la raíz o en Colab.
RAIZ = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists()), None)
if RAIZ is None:
    raise FileNotFoundError(
        "No se encontró la raíz del repositorio. Abra el notebook desde la carpeta del proyecto; "
        "en Colab, ejecute antes: !git clone <url del repositorio> y %cd <carpeta del repositorio>"
    )
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd
from src import config, validacion
from src.registro import Registro

registro = Registro("02_limpieza")
df = validacion.cargar_entrada("raw")
df_original = df.copy()

## 1. Diagnóstico inicial (Data Cleaning Nivel I y II)

Partir de las advertencias de la validación anterior. Nivel I: estructura estándar, nombres de columnas codificables y un identificador único por fila. Nivel II: reestructuración que necesite el análisis. Revisar con `info()`, `describe()` y `value_counts()` y anotar los problemas antes de corregirlos.

In [ ]:
# Integrante 2: diagnóstico (tipos, rangos, valores únicos, problemas detectados)

## 2. Valores faltantes (Nivel III)

Detección con `isnull().sum()` o `print_MVs(df)`. El dataset inicial no tiene nulos explícitos: revisar también valores vacíos disfrazados (texto vacío, guiones, "S/N"). Si aparecen, diagnosticar su tipo (MCAR, MAR o MNAR) con `analyze_numMV`, `analyze_categMV` y `chi2_contingency`, y manejarlos con alguna de estas estrategias: ignorar, `dropna()` o imputar (`fillna`, `interpolate`).

In [ ]:
# Integrante 2: detección y manejo de valores faltantes
# registro.agregar(paso="Valores faltantes", accion="...", filas_afectadas=0, justificacion="...")

## 3. Valores atípicos (Nivel III)

Univariados en `remuneracion_pen` y `vacantes` con `histplot`, `boxplot` y los límites `lowercap = Q1 - 1.5*IQR` y `uppercap = Q3 + 1.5*IQR`; bivariados con `scatterplot`; multivariados con K-means o distancia de Mahalanobis. Manejo: eliminarlos con `drop` o reemplazarlos por los límites (solo univariados). Si se fija un umbral mínimo de remuneración, registrarlo en `config.UMBRAL_REMUNERACION_MINIMA` y justificarlo.

In [ ]:
# Integrante 2: detección y manejo de outliers

## 4. Duplicados e inconsistencias

Construir una clave con `apply` (por ejemplo, entidad + número de convocatoria + puesto), contar con `value_counts()` y eliminar las repeticiones con `drop`. Revisar también textos inconsistentes y registros incoherentes (un puesto que es solo un número).

In [ ]:
# Integrante 2: duplicados, inconsistencias y datos incorrectos

## 5. Registro de decisiones

Cada decisión queda en `docs/decisiones/02_limpieza.csv` y pasa al informe.

In [ ]:
registro.tabla()

## 6. Guardado y validación

Guarda el dataset limpio y verifica el contrato `limpio`. Si la validación falla, el reporte indica qué falta corregir.

In [ ]:
print(f"Filas: {len(df_original)} → {len(df)} ({len(df_original) - len(df)} eliminadas)")
config.asegurar_carpetas()
df.to_csv(config.ARCHIVO_LIMPIO, index=False, encoding=config.ENCODING)
registro.guardar()
validacion.validar("limpio").exigir();